In [41]:
import os
import optuna 
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from optuna.integration import MLflowCallback
from collections import defaultdict
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, log_loss, f1_score
from statistics import median

In [72]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

ARTIFACT_STORE = "s3://s3-student-mle-20260102-1e8028f45c-freetrack"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000


mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")



In [73]:
EXPERIMENT_NAME = 'churn_troston'
RUN_NAME = "model_bayesian_search"

STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "bayes_hyperpar"


In [21]:
df = pd.read_csv('processed_churn.csv')
y = df['target']
X = df.drop(columns=['target', 'Unnamed: 0'])
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.1, random_state=42)

In [74]:

def objective(trial: optuna.Trial) -> float:
    param = {
   "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
   "depth": trial.suggest_int("depth", 1, 12),
   "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 5),
   "random_strength": trial.suggest_float("random_strength", 0.1, 5),
   "loss_function": "Logloss",
   "task_type": "CPU",
   "random_seed": 0,
   "iterations": 300,
   "verbose": False,
 }
 
    model = CatBoostClassifier(**param)

    skf = StratifiedKFold(n_splits=2)

    metrics = defaultdict(list)
    for i, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        train_x = X_train.iloc[train_index]
        val_x =  X_train.iloc[val_index]
        train_y = y_train.iloc[train_index]
        val_y  = y_train.iloc[val_index]
        
      
        model.fit(train_x, train_y, verbose=False)
        
        
        prediction = model.predict(val_x)
        probas = model.predict_proba(val_x)[:, 1]
        
       
        tn, fp, fn, tp = confusion_matrix(val_y, prediction).ravel()
        
    
        err1 = fp / (fp + tn) if (fp + tn) > 0 else 0  
        err2 = fn / (fn + tp) if (fn + tp) > 0 else 0  
        
        auc = roc_auc_score(val_y, probas)
        precision = precision_score(val_y, prediction)
        recall = recall_score(val_y, prediction)
        f1 = f1_score(val_y, prediction)
        logloss = log_loss(val_y, probas)
        
        metrics["err1"].append(err1)
        metrics["err2"].append(err2)
        metrics["auc"].append(auc)
        metrics["precision"].append(precision)
        metrics["recall"].append(recall)
        metrics["f1"].append(f1)
        metrics["logloss"].append(logloss)


    
    err_1 = median(np.array(metrics['err1']))
    err_2 = median(np.array(metrics['err2']))
    auc = median(np.array(metrics['auc']))
    precision = median(np.array(metrics['precision']))
    recall = median(np.array(metrics['recall']))
    f1 = median(np.array(metrics['f1']))
    logloss = median(np.array(metrics['logloss']))
    
   
    trial.set_user_attr("err1", err1)
    trial.set_user_attr("err2", err2)
    trial.set_user_attr("auc", auc)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("logloss", logloss)

    return auc

In [79]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME, artifact_location = ARTIFACT_STORE)
else:
    experiment_id = experiment.experiment_id
    

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
   
    mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="AUC",
    create_experiment=False,
    mlflow_kwargs={
        'experiment_id': experiment_id, 
        'tags': {'parent_run_id': run_id},
        'nested': True
    }
)
    
  
    study = optuna.create_study(
        study_name=STUDY_NAME, 
        storage=STUDY_DB_NAME,
        direction="maximize", 
        sampler=optuna.samplers.TPESampler(),
        load_if_exists=True    
    )
    
    study.optimize(objective, n_trials=10, callbacks=[mlflc])
    
    best_params = study.best_params
    best_value = study.best_value
    

    best_model = CatBoostClassifier(**best_params, verbose=False)
    best_model.fit(X_train, y_train)

    REGISTRY_MODEL_NAME = 'bayesian_hyperopt'
    pip_requirements= "requirements.txt"
    prediction = best_model.predict(X_test)
    prob = best_model.predict_proba(X_test)[:,1]
    signature = mlflow.models.infer_signature(X_test, prediction)
    input_example = X_train[:10]
    metadata = {'model_type': 'monthly'}
    
 
    model_info = mlflow.catboost.log_model(
        cb_model=best_model,
        artifact_path='',
        pip_requirements=pip_requirements,
        signature=signature,
        input_example=input_example,
        metadata=metadata,
        await_registration_for=60,
        registered_model_name=REGISTRY_MODEL_NAME)
    mlflow.log_params(best_params)
    mlflow.log_metric("best_auc", best_value)




print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params}")
print('run_id' , run_id)

/tmp/ipykernel_3087/2600706496.py:12: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-04-22 10:15:35,514] Using an existing study with name 'bayes_hyperpar' instead of creating a new one.


[I 2026-04-22 10:15:36,688] Trial 114 finished with value: 0.8177555604130208 and parameters: {'learning_rate': 0.02157044166625948, 'depth': 3, 'l2_leaf_reg': 4.8827164643676415, 'random_strength': 1.5925793732358224}. Best is trial 79 with value: 0.8187883536672748.
[I 2026-04-22 10:15:37,445] Trial 115 finished with value: 0.8139838236920618 and parameters: {'learning_rate': 0.02697590162887445, 'depth': 1, 'l2_leaf_reg': 4.293794175921479, 'random_strength': 2.2028437909383047}. Best is trial 79 with value: 0.8187883536672748.
[I 2026-04-22 10:15:38,235] Trial 116 finished with value: 0.8175052723638876 and parameters: {'learning_rate': 0.032234421334737876, 'depth': 2, 'l2_leaf_reg': 4.563983103596525, 'random_strength': 1.1767805116909322}. Best is trial 79 with value: 0.8187883536672748.
[I 2026-04-22 10:15:38,975] Trial 117 finished with value: 0.8177919948490879 and parameters: {'learning_rate': 0.028532466219566475, 'depth': 2, 'l2_leaf_reg': 4.129380041511495, 'random_streng

Number of finished trials: 124
Best params: {'learning_rate': 0.030644324354614724, 'depth': 2, 'l2_leaf_reg': 4.269988031611409, 'random_strength': 2.0408287820530964}
run_id 0baf8b6aaf8d4a62be3423bed1b39a05


Created version '7' of model 'bayesian_hyperopt'.


In [78]:
import boto3

s3_client = boto3.client('s3')
bucket = 's3-student-mle-20260102-1e8028f45c-freetrack'
prefix = '2/50e5d5f1066f4989961b654244cdd70f/'

response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get('Contents', []):
    print(obj['Key'])

ClientError: An error occurred (InvalidAccessKeyId) when calling the ListObjectsV2 operation: The AWS Access Key Id you provided does not exist in our records.